In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l3.2 Scaled dot-product

The core formula is `softmax(QK^T / sqrt(d)) V`. Build it by hand, then check it
against the engine primitive.

In [ ]:
from torch.nn import functional as F
from pocketlm import scaled_dot_product_attention

torch.manual_seed(0)
q = torch.randn(1, 4, 16)
k = torch.randn(1, 4, 16)
v = torch.randn(1, 4, 16)
att = (q @ k.transpose(-2, -1)) / (16 ** 0.5)
w_hand = F.softmax(att, dim=-1)
out_hand = w_hand @ v
out, w = scaled_dot_product_attention(q, k, v)
print("weights match:", torch.allclose(w, w_hand))
print("output match: ", torch.allclose(out, out_hand))

Why divide by sqrt(d)? Without it, dot products grow with dimension, the
softmax saturates to near one-hot, and gradients vanish. The scale keeps
attention smooth and trainable.

In [ ]:
assert torch.allclose(w, w_hand)
assert torch.allclose(out, out_hand)